# Bounded GA vs equal-budget random search

Bu notebook yalnız dondurulmuş proxy train/validation üzerinde 12 GA ve 12 random-search trial çalıştırır. Test görüntüleri açılmaz ve bu çalışma final performans sonucu değildir. Önce ilk üç kod hücresini çalıştırıp preflight çıktısını Codex ile incele. Uzun arama, `AUTHORIZE_GA_RANDOM_SEARCH = True` yapılmadan başlamaz.

In [ ]:
from pathlib import Path
import csv, hashlib, json, shutil, subprocess, sys, time, zipfile

assert 'google.colab' in sys.modules, 'Bu notebook Google Colab için hazırlanmıştır.'
from google.colab import drive, files
drive.mount('/content/drive')
print('Drive bağlandı.')

## 1. Repo, veri ve dondurulmuş kanıtlar
Dataset ZIP yalnız Colab'ın geçici diskine çıkarılır. Search checkpointleri ve sonuçlar Drive'da kalır.

In [ ]:
REPOSITORY_URL = 'https://github.com/YusufOck/Neural-Network-and-Genens-s-Algor-thm-Project-Erasmus-.git'
PROJECT_ROOT = Path('/content/NNGA')
ZIP_PATH = Path('/content/drive/MyDrive/NNGA_DATA/Multi Cancer.zip')
LOCAL_DATA_BASE = Path('/content/NNGA_DATA')
EXTRACT_MARKER = LOCAL_DATA_BASE / '.multi_cancer_extract_complete'
INIT_CHECKPOINT = Path('/content/drive/MyDrive/NNGA_OUTPUTS/resnet18_26leaf_baseline_v1/best_model.pth')
OUTPUT_DIR = Path('/content/drive/MyDrive/NNGA_OUTPUTS/ga_resnet18_lastblock_search_v1')
TOP_LOG = Path('/content/drive/MyDrive/NNGA_OUTPUTS/ga_resnet18_lastblock_search_v1_console.log')
REVIEW_ZIP = Path('/content/drive/MyDrive/NNGA_OUTPUTS/ga_resnet18_lastblock_search_v1_review_package.zip')
FROZEN_SPLIT_SHA256 = '999bf7a9bd1bba9f2957a9db0e13ae72adb3ecb8fdbc1f33cce40b083c1cf171'
FROZEN_PROXY_SHA256 = 'fd6a55def92aa5f3582679ef0be35bde6fcb722be00fb508db9a86a7030aff03'
FROZEN_INIT_SHA256 = '20613069067825ea0dfa30e2242678b863cd0acd7b9be3c5b88f265d39285177'
FROZEN_TRAINER_SHA256 = '807a23e40ea3aaa5c0c98737beddb637d71c5e5a9e5f230aa0b75d0be7212a77'
FROZEN_RUNNER_SHA256 = '657190d6b61ec1a1ce7a7aba6b5345e4fbcee376431a1ea9825d3b1dc4c1df7d'
FROZEN_CONFIG_SHA256 = 'fefe845a8fb84c0f252954f42cee3a8f4fa84210a4313d24af339fdeb90c7638'
EXPECTED_GROUPS = ('ALL', 'Brain Cancer', 'Breast Cancer', 'Cervical Cancer', 'Kidney Cancer', 'Lung and Colon Cancer', 'Lymphoma', 'Oral Cancer')
EXPECTED_CLASS_NAMES = ('all_benign', 'all_early', 'all_pre', 'all_pro', 'brain_glioma', 'brain_menin', 'brain_tumor', 'breast_benign', 'breast_malignant', 'cervix_dyk', 'cervix_koc', 'cervix_mep', 'cervix_pab', 'cervix_sfi', 'colon_aca', 'colon_bnt', 'kidney_normal', 'kidney_tumor', 'lung_aca', 'lung_bnt', 'lung_scc', 'lymph_cll', 'lymph_fl', 'lymph_mcl', 'oral_normal', 'oral_scc')

if (PROJECT_ROOT / '.git').is_dir():
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
commit = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
SPLIT_MANIFEST = PROJECT_ROOT / 'research/splits/practical_leakage_reduced_split_v1.csv'
PROXY_MANIFEST = PROJECT_ROOT / 'research/splits/ga_search_proxy_v1.csv'
PROXY_REPORT = PROJECT_ROOT / 'research/splits/ga_search_proxy_v1_report.json'
CONFIG_PATH = PROJECT_ROOT / 'config/ga_resnet18_lastblock_v1.json'
RUNNER_PATH = PROJECT_ROOT / 'src/run_ga_random_search.py'
TRAINER_PATH = PROJECT_ROOT / 'src/train.py'
for path, expected in ((SPLIT_MANIFEST, FROZEN_SPLIT_SHA256), (PROXY_MANIFEST, FROZEN_PROXY_SHA256), (INIT_CHECKPOINT, FROZEN_INIT_SHA256), (TRAINER_PATH, FROZEN_TRAINER_SHA256), (RUNNER_PATH, FROZEN_RUNNER_SHA256), (CONFIG_PATH, FROZEN_CONFIG_SHA256)):
    assert path.is_file(), f'Eksik frozen girdi: {path}'
    actual = hashlib.sha256(path.read_bytes()).hexdigest()
    assert actual == expected, f'Hash uyuşmuyor: {path} expected={expected} actual={actual}'
assert ZIP_PATH.is_file(), f'Dataset ZIP bulunamadı: {ZIP_PATH}'
free_gib = shutil.disk_usage('/content').free / (1024 ** 3)
assert free_gib >= 12.0, 'Dataset açma ve search için en az 12 GiB yerel boş alan gerekli.'
if not EXTRACT_MARKER.is_file():
    LOCAL_DATA_BASE.mkdir(parents=True, exist_ok=True)
    started = time.monotonic()
    print('ZIP Colab diskine çıkarılıyor; birkaç dakika çıktı görünmeyebilir...')
    subprocess.run(['unzip', '-q', '-o', str(ZIP_PATH), '-d', str(LOCAL_DATA_BASE)], check=True)
    EXTRACT_MARKER.write_text('complete\n', encoding='utf-8')
    print(f'ZIP çıkarma tamamlandı: {(time.monotonic() - started) / 60:.1f} dakika')
dataset_candidates = (LOCAL_DATA_BASE / 'Multi Cancer' / 'Multi Cancer', LOCAL_DATA_BASE / 'Multi Cancer', LOCAL_DATA_BASE)
DATASET_ROOT = next((p for p in dataset_candidates if all((p / g).is_dir() for g in EXPECTED_GROUPS)), None)
assert DATASET_ROOT is not None, 'Sekiz ana klasörü içeren dataset kökü bulunamadı.'
OUTPUT_DIR.parent.mkdir(parents=True, exist_ok=True)
print('Git commit:', commit)
print('Dataset:', DATASET_ROOT)
print('Split SHA-256:', FROZEN_SPLIT_SHA256)
print('Proxy SHA-256:', FROZEN_PROXY_SHA256)
print('Init checkpoint SHA-256:', FROZEN_INIT_SHA256)
print('Search output:', OUTPUT_DIR)

## 2. GPU ve search sözleşmesi preflight
Bu hücre model eğitmez. Proxy sayımlarını, test yasağını, 12+12 bütçeyi ve layer4+fc kapsamını doğrular.

In [ ]:
import torch, torchvision
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from train import build_model, validate_initial_checkpoint
from run_ga_random_search import validate_config
assert torch.cuda.is_available(), 'GPU seçilmemiş; T4 runtime aç.'
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
validate_config(config)
proxy_report = json.loads(PROXY_REPORT.read_text(encoding='utf-8'))
assert proxy_report['counts'] == {'train': 9100, 'val': 3900, 'test': 0}
assert proxy_report['leaf_count'] == 26
assert all(counts == {'train': 350, 'val': 150} for counts in proxy_report['per_leaf'].values())
assert config['equal_budget'] == {'ga_trials': 12, 'random_trials': 12, 'population_size': 4, 'ga_rounds': 3}
assert config['test_policy']['test_opened_during_search'] is False
checkpoint, checked_hash = validate_initial_checkpoint(INIT_CHECKPOINT, FROZEN_INIT_SHA256, 'resnet18', 'subclass', EXPECTED_CLASS_NAMES)
probe = build_model('resnet18', 26, pretrained=True, fine_tune=True, dropout=0.30)
probe.load_state_dict(checkpoint['model_state_dict'], strict=True)
trainable = {name: p.numel() for name, p in probe.named_parameters() if p.requires_grad}
assert trainable and all(name.startswith(('layer4.', 'fc.')) for name in trainable)
assert sum(trainable.values()) == 8407066
del probe, checkpoint
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__, '| torchvision:', torchvision.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('Trainable scope: yalnız layer4 + fc | parameter:', sum(trainable.values()))
print('Proxy: 9.100 train / 3.900 validation / 0 test')
print('Bütçe: 12 GA + 12 random; aday başına 4 epoch')
print('Preflight başarılı. Henüz eğitim başlamadı.')

## 3. Kilitli uzun search
İlk üç kod hücresinin çıktısını Codex ile incelemeden aşağıdaki değeri değiştirme. Runtime kesilirse aynı hücre yeniden çalıştırılır; tamamlanmış trial'lar atlanır, yarım trial checkpointten devam eder.

In [ ]:
AUTHORIZE_GA_RANDOM_SEARCH = False
assert AUTHORIZE_GA_RANDOM_SEARCH, 'GA/random search kilitli; preflight Codex ile onaylanmadan çalışmaz.'
summary_path = OUTPUT_DIR / 'search_summary.json'
if summary_path.is_file():
    existing_summary = json.loads(summary_path.read_text(encoding='utf-8'))
    assert existing_summary.get('status') != 'ga_random_search_v1_complete', 'Search zaten tamamlanmış; yeniden çalıştırma.'
command = [
    sys.executable, '-u', str(RUNNER_PATH),
    '--data-dir', str(DATASET_ROOT),
    '--split-manifest', str(SPLIT_MANIFEST),
    '--proxy-manifest', str(PROXY_MANIFEST),
    '--init-checkpoint', str(INIT_CHECKPOINT),
    '--output-dir', str(OUTPUT_DIR),
    '--config', str(CONFIG_PATH),
    '--workers', '4', '--method', 'both',
    '--authorize', 'FROZEN_GA_RANDOM_SEARCH_V1',
]
log_mode = 'a' if TOP_LOG.exists() else 'w'
print('Search komutu:', ' '.join(command))
with TOP_LOG.open(log_mode, encoding='utf-8') as log_handle:
    process = subprocess.Popen(command, cwd=PROJECT_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace', bufsize=1)
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        log_handle.write(line); log_handle.flush()
    return_code = process.wait()
print('RETURN CODE:', return_code)
if return_code != 0:
    raise RuntimeError('Search tamamlanamadı. Output klasörünü silme; logu Codex’e gönder.')

## 4. Tamamlanmış search paketini doğrula ve indir
Bu hücre yalnız 12+12 trial tamamlandıktan sonra çalışır. Checkpointleri ZIP'e koymaz; Drive output klasöründe korur.

In [ ]:
required_root = ['run_contract.json', 'frozen_config.json', 'search_results.csv', 'search_summary.json']
missing = [name for name in required_root if not (OUTPUT_DIR / name).is_file()]
assert not missing, f'Eksik search çıktıları: {missing}'
summary = json.loads((OUTPUT_DIR / 'search_summary.json').read_text(encoding='utf-8'))
assert summary['status'] == 'ga_random_search_v1_complete'
assert summary['completed_trials'] == {'ga': 12, 'random': 12}
assert summary['expected_trials'] == {'ga': 12, 'random': 12}
assert summary['equal_budget_complete'] is True and summary['test_evaluated'] is False
with (OUTPUT_DIR / 'search_results.csv').open(encoding='utf-8', newline='') as handle:
    result_rows = list(csv.DictReader(handle))
assert len(result_rows) == 24
assert sum(row['method'] == 'ga' for row in result_rows) == 12
assert sum(row['method'] == 'random' for row in result_rows) == 12
assert all(row['test_evaluated'].lower() == 'false' for row in result_rows)
assert not list((OUTPUT_DIR / 'trials').glob('*/test_metrics.json')), 'Search trial test metriği üretmiş.'
review_paths = [OUTPUT_DIR / name for name in required_root]
if TOP_LOG.is_file(): review_paths.append(TOP_LOG)
for trial_dir in sorted((OUTPUT_DIR / 'trials').iterdir()):
    for name in ('candidate.json', 'trial_result.json', 'search_trial_summary.json', 'history.csv', 'run_config.json'):
        path = trial_dir / name
        assert path.is_file(), f'Eksik trial kanıtı: {path}'
        review_paths.append(path)
artifact_hashes = {}
for path in review_paths:
    arcname = path.name if path == TOP_LOG else path.relative_to(OUTPUT_DIR).as_posix()
    artifact_hashes[arcname] = {'sha256': hashlib.sha256(path.read_bytes()).hexdigest(), 'size_bytes': path.stat().st_size}
manifest_path = OUTPUT_DIR / 'review_artifact_hashes.json'
manifest_path.write_text(json.dumps(artifact_hashes, indent=2), encoding='utf-8')
with zipfile.ZipFile(REVIEW_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in review_paths:
        arcname = path.name if path == TOP_LOG else path.relative_to(OUTPUT_DIR).as_posix()
        archive.write(path, arcname=arcname)
    archive.write(manifest_path, arcname='review_artifact_hashes.json')
review_hash = hashlib.sha256(REVIEW_ZIP.read_bytes()).hexdigest()
print('Durum:', summary['status'])
print('Tamamlanan:', summary['completed_trials'])
print('GA kazananı:', summary['winners']['ga'])
print('Random kazananı:', summary['winners']['random'])
print('Test evaluated:', summary['test_evaluated'])
print('Review package:', REVIEW_ZIP)
print('Review package SHA-256:', review_hash)
files.download(str(REVIEW_ZIP))